# **MobileNetV2 Quantization and Layer Sensitivity**

# ECE 662 — Team 9

**Owner**: Lilly Press

**Goal**: Compare 8-bit and 4-bit weight quantization, then identify which layers lose the most accuracy when reduced to 4 bits.  

**Current stage**: Uniform weight quantization. Activations remain FP32 during initial debugging; final experiments will use fixed 8-bit activations.

# 1. Setup and Imports

In [4]:
import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

# Set seeds for reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Use a GPU if available; otherwise use the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"Device: {device}")

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.11.0+cpu
Torchvision version: 0.26.0+cpu
Device: cpu


# 2. Load the Baseline Model

In [5]:
# Load the team's shared trained model
from torchvision.models import mobilenet_v2

baseline_model = mobilenet_v2(weights=None)
baseline_model.classifier[1] = nn.Linear(
    baseline_model.last_channel, 10
)

state_dict = torch.load(
    "/content/mobilenetv2_cifar10.pth",
    map_location="cpu",
    weights_only=True
)

baseline_model.load_state_dict(state_dict, strict=True)
baseline_model = baseline_model.to(device).eval()

print("Shared checkpoint loaded successfully!")
print("Device:", device)

Shared checkpoint loaded successfully!
Device: cpu


# 3. Load CIFAR-10

In [6]:
# Convert images to tensors and normalize each color channel
# Match Josie's evaluation preprocessing
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

# Download the CIFAR-10 test set
test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# Full test set for reporting final accuracy
test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

# Small test subset for quick debugging only
generator = torch.Generator().manual_seed(SEED)
debug_indices = torch.randperm(
    len(test_dataset), generator=generator
)[:1000].tolist()

debug_loader = DataLoader(
    Subset(test_dataset, debug_indices),
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

print(f"Full test set: {len(test_dataset):,} images")
print(f"Debug subset: {len(debug_indices):,} images")
print(f"Classes: {test_dataset.classes}")

100%|██████████| 170M/170M [55:38<00:00, 51.1kB/s]


Full test set: 10,000 images
Debug subset: 1,000 images
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


# 4. Check FP32 Accuracy

In [7]:
# Reuse this function for every model configuration
def evaluate(model, loader):
    model.eval()
    model_device = next(model.parameters()).device

    correct = 0
    total = 0

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(model_device)
            labels = labels.to(model_device)

            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    if total == 0:
        raise ValueError("The data loader contains no images.")

    return 100.0 * correct / total


# Start with the smaller subset to check everything works
fp32_debug_accuracy = evaluate(baseline_model, debug_loader)
print(f"FP32 debug accuracy: {fp32_debug_accuracy:.2f}%")

FP32 debug accuracy: 89.00%


In [8]:
fp32_test_accuracy = evaluate(baseline_model, test_loader)
print(f"FP32 full-test accuracy: {fp32_test_accuracy:.2f}%")

FP32 full-test accuracy: 89.82%


# 5. Define Weight Quantization

In [9]:
def quantize_weights(weights, bits):
    """Simulate symmetric quantization per output channel."""
    if bits not in (4, 8):
        raise ValueError("bits must be 4 or 8")

    # Symmetric integer range: -7 to 7 or -127 to 127
    qmax = (2 ** (bits - 1)) - 1

    # Dimension 0 represents output channels for Conv2d and Linear.
    # Find the maximum magnitude within each output channel.
    reduce_dims = tuple(range(1, weights.ndim))
    max_abs = weights.abs().amax(dim=reduce_dims, keepdim=True)

    # Use scale 1 for all-zero channels to avoid division by zero
    scale = torch.where(
        max_abs > 0,
        max_abs / qmax,
        torch.ones_like(max_abs)
    )

    integer_weights = torch.round(weights / scale)
    integer_weights = torch.clamp(integer_weights, -qmax, qmax)

    # Convert back to floating point for ordinary PyTorch inference
    return integer_weights * scale


def quantize_model_weights(model, bits):
    """Return a quantized copy, leaving the original model unchanged."""
    quantized_model = copy.deepcopy(model)
    quantized_model.eval()

    with torch.no_grad():
        for name, layer in quantized_model.named_modules():
            if isinstance(layer, (nn.Conv2d, nn.Linear)):
                layer.weight.copy_(
                    quantize_weights(layer.weight, bits)
                )

    return quantized_model


print("Weight quantization functions are ready.")

Weight quantization functions are ready.


# 6. Run Uniform 8-bit Weights

In [19]:
# Create an 8-bit weight version from the untouched baseline
model_w8 = quantize_model_weights(baseline_model, bits=8)

# Check accuracy on the same debugging subset as FP32
w8_debug_accuracy = evaluate(model_w8, debug_loader)

print("Configuration: W8 / FP32 activations")
print(f"FP32 debug accuracy: {fp32_debug_accuracy:.2f}%")
print(f"W8 debug accuracy:   {w8_debug_accuracy:.2f}%")
print(
    f"Accuracy drop: "
    f"{fp32_debug_accuracy - w8_debug_accuracy:.2f} percentage points"
)

Configuration: W8 / FP32 activations
FP32 debug accuracy: 89.00%
W8 debug accuracy:   89.70%
Accuracy drop: -0.70 percentage points


In [20]:
w8_test_accuracy = evaluate(model_w8, test_loader)

print(f"W8 full-test accuracy: {w8_test_accuracy:.2f}%")
print(
    f"Accuracy drop from FP32: "
    f"{fp32_test_accuracy - w8_test_accuracy:.2f} percentage points"
)

W8 full-test accuracy: 89.87%
Accuracy drop from FP32: -0.05 percentage points


# 7. Run Uniform 4-bit Weights

In [21]:
# Create a fresh 4-bit version from the original FP32 model
model_w4 = quantize_model_weights(baseline_model, bits=4)

w4_debug_accuracy = evaluate(model_w4, debug_loader)

print("Configuration: W4 / FP32 activations")
print(f"FP32 debug accuracy: {fp32_debug_accuracy:.2f}%")
print(f"W4 debug accuracy:   {w4_debug_accuracy:.2f}%")
print(
    f"Accuracy drop: "
    f"{fp32_debug_accuracy - w4_debug_accuracy:.2f} percentage points"
)

Configuration: W4 / FP32 activations
FP32 debug accuracy: 89.00%
W4 debug accuracy:   19.10%
Accuracy drop: 69.90 percentage points


In [22]:
w4_test_accuracy = evaluate(model_w4, test_loader)

print(f"W4 full-test accuracy: {w4_test_accuracy:.2f}%")
print(
    f"Accuracy drop from FP32: "
    f"{fp32_test_accuracy - w4_test_accuracy:.2f} percentage points"
)

W4 full-test accuracy: 17.58%
Accuracy drop from FP32: 72.24 percentage points


# 8. Calculate Estimated Weight Storage

In [23]:
def estimate_weight_storage(model, bits):
    """Packed Conv2d/Linear weight payload only, in decimal MB."""
    weight_count = sum(
        layer.weight.numel()
        for layer in model.modules()
        if isinstance(layer, (nn.Conv2d, nn.Linear))
    )

    return weight_count * bits / 8 / 1_000_000


storage_results = pd.DataFrame({
    "Weight precision": ["FP32", "8-bit", "4-bit"],
    "Estimated weight payload (MB)": [
        estimate_weight_storage(baseline_model, 32),
        estimate_weight_storage(baseline_model, 8),
        estimate_weight_storage(baseline_model, 4)
    ]
})

print(storage_results.to_string(index=False))

Weight precision  Estimated weight payload (MB)
            FP32                        8.81024
           8-bit                        2.20256
           4-bit                        1.10128


# 9. Compare Baseline Results

In [24]:
baseline_results = pd.DataFrame({
    "Configuration": [
        "FP32",
        "W8 / FP32 activations",
        "W4 / FP32 activations"
    ],
    "Test accuracy (%)": [
        fp32_test_accuracy,
        w8_test_accuracy,
        w4_test_accuracy
    ],
    "Drop from FP32 (percentage points)": [
        0.0,
        fp32_test_accuracy - w8_test_accuracy,
        fp32_test_accuracy - w4_test_accuracy
    ],
    "Estimated weight payload (MB)": [
        estimate_weight_storage(baseline_model, 32),
        estimate_weight_storage(baseline_model, 8),
        estimate_weight_storage(baseline_model, 4)
    ]
})

display(baseline_results.round(3))

,Configuration,Test accuracy (%),Drop from FP32 (percentage points),Estimated weight payload (MB)
0,FP32,89.82,0.00,8.810
1,W8 / FP32 activations,89.87,-0.05,2.203
2,W4 / FP32 activations,17.58,72.24,1.101


# 10. Add per-layer precision

In [25]:
# Names of all layers eligible for weight quantization
eligible_layers = [
    name
    for name, layer in baseline_model.named_modules()
    if isinstance(layer, (nn.Conv2d, nn.Linear))
]


def quantize_model_by_layer(model, bit_config):
    """Quantize a fresh model copy using a layer-name-to-bits dictionary."""
    model_layers = {
        name: layer
        for name, layer in model.named_modules()
        if isinstance(layer, (nn.Conv2d, nn.Linear))
    }

    # Require a precision assignment for every eligible layer
    if set(bit_config) != set(model_layers):
        raise ValueError(
            "bit_config must contain exactly the eligible layer names."
        )

    if any(bits not in (4, 8) for bits in bit_config.values()):
        raise ValueError("Each layer must use 4 or 8 bits.")

    quantized_model = copy.deepcopy(model)
    quantized_model.eval()

    with torch.no_grad():
        for name, layer in quantized_model.named_modules():
            if name in bit_config:
                layer.weight.copy_(
                    quantize_weights(layer.weight, bit_config[name])
                )

    return quantized_model


# Start with every eligible layer at 8 bits
all_w8_config = {name: 8 for name in eligible_layers}

# Example: change only the first eligible layer to 4 bits
example_config = all_w8_config.copy()
example_config[eligible_layers[0]] = 4

print(f"Eligible layers: {len(eligible_layers)}")
print(f"Example 4-bit layer: {eligible_layers[0]}")
print("Per-layer quantization function is ready.")

Eligible layers: 53
Example 4-bit layer: features.0.0
Per-layer quantization function is ready.


In [26]:
# Build an all-8-bit model using the per-layer configuration
model_w8_by_layer = quantize_model_by_layer(
    baseline_model, all_w8_config
)

# Its weights and other state should match the existing W8 model
for name, value in model_w8.state_dict().items():
    assert torch.equal(
        value, model_w8_by_layer.state_dict()[name]
    ), f"Mismatch in: {name}"

print("Passed: per-layer all-W8 matches the uniform W8 model.")

# Check that the example mixed configuration runs
example_mixed_model = quantize_model_by_layer(
    baseline_model, example_config
)

example_debug_accuracy = evaluate(
    example_mixed_model, debug_loader
)

print(f"4-bit layer: {eligible_layers[0]}")
print(f"All other eligible layers: 8 bits")
print(f"Debug accuracy: {example_debug_accuracy:.2f}%")

Passed: per-layer all-W8 matches the uniform W8 model.
4-bit layer: features.0.0
All other eligible layers: 8 bits
Debug accuracy: 87.90%


# 11. Fold Batch Normalization

In [27]:
from torch.nn.utils import fuse_conv_bn_eval

def fold_batch_norm(model):
    folded_model = copy.deepcopy(model).eval()
    folded_pairs = []

    def fold_sequential_pairs(module, prefix=""):
        # Recurse into nested modules
        for name, child in list(module.named_children()):
            child_prefix = f"{prefix}.{name}" if prefix else name
            fold_sequential_pairs(child, child_prefix)

        # Only combine adjacent layers in Sequential containers
        if isinstance(module, nn.Sequential):
            children = list(module.named_children())

            for i in range(len(children) - 1):
                conv_name, conv = children[i]
                bn_name, bn = children[i + 1]

                if isinstance(conv, nn.Conv2d) and isinstance(
                    bn, nn.BatchNorm2d
                ):
                    module._modules[conv_name] = fuse_conv_bn_eval(
                        conv, bn
                    )
                    module._modules[bn_name] = nn.Identity()
                    folded_pairs.append(
                        f"{prefix}.{conv_name}" if prefix else conv_name
                    )

    fold_sequential_pairs(folded_model)

    remaining_bn = sum(
        isinstance(layer, nn.BatchNorm2d)
        for layer in folded_model.modules()
    )
    if remaining_bn:
        raise ValueError(
            f"{remaining_bn} batch-normalization layers remain; "
            "check the model structure."
        )

    return folded_model, folded_pairs


# Preserve the original baseline
folded_baseline, folded_pairs = fold_batch_norm(baseline_model)

# Compare outputs on one real image batch
images, _ = next(iter(debug_loader))
images = images.to(next(baseline_model.parameters()).device)

with torch.inference_mode():
    original_outputs = baseline_model(images)
    folded_outputs = folded_baseline(images)

max_difference = (
    original_outputs - folded_outputs
).abs().max().item()

torch.testing.assert_close(
    folded_outputs,
    original_outputs,
    rtol=1e-3,
    atol=1e-4
)

print(f"Folded convolution/BN pairs: {len(folded_pairs)}")
print(f"Maximum output difference: {max_difference:.8f}")
print("Passed: folding preserves FP32 outputs within tolerance.")

Folded convolution/BN pairs: 52
Maximum output difference: 0.00004005
Passed: folding preserves FP32 outputs within tolerance.


# 12. Evaluate Folded Weight-Only Baselines

In [28]:
# Keep earlier models and results unchanged.
# Quantize fresh copies AFTER BatchNorm folding.
folded_w8_model = quantize_model_weights(folded_baseline, bits=8)
folded_w4_model = quantize_model_weights(folded_baseline, bits=4)

folded_fp32_accuracy = evaluate(folded_baseline, test_loader)
folded_w8_accuracy = evaluate(folded_w8_model, test_loader)
folded_w4_accuracy = evaluate(folded_w4_model, test_loader)

folded_results = pd.DataFrame({
    "Configuration": [
        "Folded FP32",
        "Folded W8 / FP32 activations",
        "Folded W4 / FP32 activations"
    ],
    "Test accuracy (%)": [
        folded_fp32_accuracy,
        folded_w8_accuracy,
        folded_w4_accuracy
    ],
    "Drop from folded FP32 (percentage points)": [
        0.0,
        folded_fp32_accuracy - folded_w8_accuracy,
        folded_fp32_accuracy - folded_w4_accuracy
    ]
})

display(folded_results.round(3))

,Configuration,Test accuracy (%),Drop from folded FP32 (percentage points)
0,Folded FP32,89.82,0.00
1,Folded W8 / FP32 activations,89.87,-0.05
2,Folded W4 / FP32 activations,17.58,72.24


# Prepare Validation Data for Layer Sensitivity

In [29]:
# Reproduce Josie's held-out validation split:
# seeded permutation, first 45,000 training, last 5,000 validation.
validation_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

split_indices = torch.randperm(
    len(validation_dataset),
    generator=torch.Generator().manual_seed(42)
)

validation_indices = split_indices[45000:].tolist()

validation_loader = DataLoader(
    Subset(validation_dataset, validation_indices),
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

print("Validation images:", len(validation_indices))

Validation images: 5000


# 14. Run Layer Sensitivity Sweep

In [31]:
import time

# Rank layers using validation data only.
# Activations remain FP32 in this experiment.
sensitivity_layers = [
    name for name, layer in folded_baseline.named_modules()
    if isinstance(layer, (nn.Conv2d, nn.Linear))
]

sweep_w8_config = {name: 8 for name in sensitivity_layers}

w8_validation_accuracy = evaluate(
    folded_w8_model, validation_loader
)
print(f"W8 validation accuracy: {w8_validation_accuracy:.2f}%")

sensitivity_rows = []
start_time = time.time()

for i, layer_name in enumerate(sensitivity_layers, start=1):
    # Start fresh: all W8 except this one layer at W4.
    config = sweep_w8_config.copy()
    config[layer_name] = 4

    candidate = quantize_model_by_layer(
        folded_baseline, config
    )
    accuracy = evaluate(candidate, validation_loader)
    drop = w8_validation_accuracy - accuracy

    sensitivity_rows.append({
        "Layer": layer_name,
        "Validation accuracy (%)": accuracy,
        "Drop from W8 (percentage points)": drop
    })

    # Save progress after every layer.
    pd.DataFrame(sensitivity_rows).to_csv(
        "/content/layer_sensitivity_progress.csv", index=False
    )

    print(
        f"[{i}/{len(sensitivity_layers)}] {layer_name}: "
        f"{accuracy:.2f}% | drop {drop:.2f} pp",
        flush=True
    )
    del candidate

sensitivity_results = (
    pd.DataFrame(sensitivity_rows)
    .sort_values(
        "Drop from W8 (percentage points)",
        ascending=False,
        kind="stable"
    )
    .reset_index(drop=True)
)

sensitivity_results.insert(
    0, "Rank", range(1, len(sensitivity_results) + 1)
)

sensitivity_results.to_csv(
    "/content/layer_sensitivity_ranking.csv", index=False
)

print(f"\nFinished in {(time.time() - start_time) / 60:.1f} minutes")
display(sensitivity_results.head(10))

W8 validation accuracy: 89.98%
[1/53] features.0.0: 87.10% | drop 2.88 pp
[2/53] features.1.conv.0.0: 71.74% | drop 18.24 pp
[3/53] features.1.conv.1: 78.98% | drop 11.00 pp
[4/53] features.2.conv.0.0: 89.66% | drop 0.32 pp
[5/53] features.2.conv.1.0: 75.02% | drop 14.96 pp
[6/53] features.2.conv.2: 86.60% | drop 3.38 pp
[7/53] features.3.conv.0.0: 89.80% | drop 0.18 pp
[8/53] features.3.conv.1.0: 87.00% | drop 2.98 pp
[9/53] features.3.conv.2: 88.82% | drop 1.16 pp
[10/53] features.4.conv.0.0: 89.78% | drop 0.20 pp
[11/53] features.4.conv.1.0: 89.96% | drop 0.02 pp
[12/53] features.4.conv.2: 87.74% | drop 2.24 pp
[13/53] features.5.conv.0.0: 89.68% | drop 0.30 pp
[14/53] features.5.conv.1.0: 89.34% | drop 0.64 pp
[15/53] features.5.conv.2: 88.62% | drop 1.36 pp
[16/53] features.6.conv.0.0: 89.96% | drop 0.02 pp
[17/53] features.6.conv.1.0: 89.66% | drop 0.32 pp
[18/53] features.6.conv.2: 89.48% | drop 0.50 pp
[19/53] features.7.conv.0.0: 90.18% | drop -0.20 pp
[20/53] features.7.conv.

,Rank,Layer,Validation accuracy (%),Drop from W8 (percentage points)
0,1,features.1.conv.0.0,71.74,18.24
1,2,features.2.conv.1.0,75.02,14.96
2,3,features.1.conv.1,78.98,11.00
3,4,features.2.conv.2,86.60,3.38
4,5,features.3.conv.1.0,87.00,2.98
5,6,features.0.0,87.10,2.88
6,7,features.7.conv.2,87.38,2.60
7,8,features.4.conv.2,87.74,2.24
8,9,features.14.conv.2,88.38,1.60
9,10,features.5.conv.2,88.62,1.36


# 15. Evaluate Mixed-Precision Configurations

In [32]:
import json

ranked_layers = sensitivity_results["Layer"].tolist()
folded_layers = dict(folded_baseline.named_modules())

mixed_rows = []
mixed_configs = {}

for k in [0, 1, 3, 5, 10]:
    # Protect the top k layers at W8; all others use W4.
    protected_layers = set(ranked_layers[:k])
    config = {
        name: 8 if name in protected_layers else 4
        for name in sensitivity_layers
    }
    mixed_configs[f"k_{k}"] = config

    mixed_model = quantize_model_by_layer(
        folded_baseline, config
    )
    accuracy = evaluate(mixed_model, test_loader)

    # Theoretical Conv/Linear weight payload only.
    payload_mb = sum(
        folded_layers[name].weight.numel() * bits
        for name, bits in config.items()
    ) / 8 / 1_000_000

    mixed_rows.append({
        "Protected W8 layers (k)": k,
        "Test accuracy (%)": accuracy,
        "Drop from folded W8 (pp)": folded_w8_accuracy - accuracy,
        "Estimated weight payload (MB)": payload_mb
    })

    pd.DataFrame(mixed_rows).to_csv(
        "/content/mixed_precision_results.csv", index=False
    )

    print(f"k={k}: {accuracy:.2f}% | {payload_mb:.4f} MB",
          flush=True)
    del mixed_model

with open("/content/mixed_precision_configs.json", "w") as f:
    json.dump(mixed_configs, f, indent=2)

mixed_results = pd.DataFrame(mixed_rows)
display(mixed_results.round(3))

k=0: 17.58% | 1.1013 MB
k=1: 26.78% | 1.1014 MB
k=3: 58.91% | 1.1021 MB
k=5: 63.99% | 1.1039 MB
k=10: 79.88% | 1.1619 MB


,Protected W8 layers (k),Test accuracy (%),Drop from folded W8 (pp),Estimated weight payload (MB)
0,0,17.58,72.29,1.101
1,1,26.78,63.09,1.101
2,3,58.91,30.96,1.102
3,5,63.99,25.88,1.104
4,10,79.88,9.99,1.162
